# 04 – LSTM Model

This notebook trains a multi-layer LSTM network on the scaled, feature-engineered
dataset and evaluates it on the test set.


In [1]:
# ── 1. Imports & Configuration ──────────────────────────────────────────────
import sys

sys.path.insert(0, '..')

import warnings

warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.data.preprocessor import DataPreprocessor
from src.models.lstm_model import LSTMModel
from src.visualization.plotter import Plotter


from src.config import RESULTS_DIR, LSTM_SEQUENCE_LENGTH

plotter = Plotter()
SEQ_LEN = LSTM_SEQUENCE_LENGTH  # LSTM的时间步长

I0000 00:00:1774181594.395705  101639 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774181594.445794  101639 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774181596.693930  101639 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# ── 2. Load Data & Scale ────────────────────────────────────────────────────
train = load_processed_data('train')
val = load_processed_data('val')
test = load_processed_data('test')



# 提取并保存真实的未归一化的收盘价，用于后面算误差
y_test_val_df = pd.concat([val, test])

# 归一化特征
pre = DataPreprocessor()
t_sc, v_sc, te_sc = pre.scale_features(train, val, test)

target_col = 'Close'
target_idx = list(train.columns).index(target_col)

print(f'Target column index: {target_idx}')
print(f'Scaled shapes – train: {t_sc.shape}, val: {v_sc.shape}, test: {te_sc.shape}')
print(f'Target column index: {target_idx}')
print(f'Scaled shapes – train: {t_sc.shape}, val: {v_sc.shape}, test: {te_sc.shape}')
print(f'Columns used ({len(train.columns)}): {list(train.columns)}')

Target column index: 1
Scaled shapes – train: (1006, 32), val: (252, 32), test: (249, 32)
Target column index: 1
Scaled shapes – train: (1006, 32), val: (252, 32), test: (249, 32)
Columns used (32): ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'MA5', 'MA10', 'MA20', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_upper', 'BB_lower', 'BB_mid', 'BB_pct', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'Volume_MA5', 'Volume_MA20', 'OBV', 'GDP_Lag35', 'CPI_Lag35', 'FED_RATE_Lag35', 'UNRATE_Lag35', 'sentiment_pos', 'sentiment_neg', 'sentiment_compound']


In [3]:
# ── 3. Create LSTM Sequences ────────────────────────────────────────────────
def create_sequences(data, target_index, seq_length):
    #  Pandas 格式，强制转换为 numpy 纯数组格式
    if isinstance(data, (pd.DataFrame, pd.Series)):
        data = data.values

    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i: i + seq_length])
        y.append(data[i + seq_length, target_index])
    return np.array(X), np.array(y)


# 传入数据进行切片
X_train, y_train = create_sequences(t_sc, target_idx, SEQ_LEN)
X_val, y_val = create_sequences(v_sc, target_idx, SEQ_LEN)
X_test, y_test = create_sequences(te_sc, target_idx, SEQ_LEN)

print(f"X_train shape: {X_train.shape} (Samples, TimeSteps, Features)")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (946, 60, 32) (Samples, TimeSteps, Features)
X_val shape:   (192, 60, 32)
X_test shape:  (189, 60, 32)


In [4]:
# ── 4. Train, Evaluate & Plot LSTM ──────────────────────────────────────────
lstm = LSTMModel()

train_idx = train.index[SEQ_LEN:] if len(train.index[SEQ_LEN:]) == len(X_train) else train.index[-len(X_train):]
val_idx = val.index[SEQ_LEN:] if len(val.index[SEQ_LEN:]) == len(X_val) else val.index[-len(X_val):]
test_idx = test.index[SEQ_LEN:] if len(test.index[SEQ_LEN:]) == len(X_test) else test.index[-len(X_test):]
# 1. 获取被归一化压缩的预测值
lstm_val_preds_scaled, lstm_test_preds_scaled = lstm.train_and_refit(
    X_train, y_train, X_val, y_val, X_test,
    train_index=train_idx,
    val_index=val_idx,
    test_index=test_idx,
    phase2_start_date="2021-01-01",
    phase2_split_date="2024-10-01",
)

print("--- Training LSTM (Two-Phase Refitting) ---")
# =========================================================================
# 绘制并保存训练 Loss 曲线
# =========================================================================
param_tag = (
    f"{SEQ_LEN}_{lstm.dropout}_{lstm.learning_rate:.0e}"
    .replace("e-0", "e-")
    .replace("e+0", "e+")
)

plt.figure(figsize=(10, 5))
plt.plot(lstm.history.history['loss'], label='Training Loss (MSE)', color='#1f77b4', linewidth=2)
plt.plot(lstm.history.history['val_loss'], label='Validation Loss (MSE)', color='#ff7f0e', linewidth=2)

plt.title(f"LSTM Refitting Phase - Loss Curve ({param_tag})", fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)

figures_dir = RESULTS_DIR.parent / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
loss_path = figures_dir / f"lstm_training_loss_{param_tag}.png"
plt.savefig(loss_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Training Loss chart saved to: {loss_path}")

# =========================================================================
# 反归一化 Inverse Transform (还原真实股价)
# =========================================================================
dummy_val = np.zeros((len(lstm_val_preds_scaled), len(train.columns)))
dummy_test = np.zeros((len(lstm_test_preds_scaled), len(train.columns)))

dummy_val[:, target_idx] = lstm_val_preds_scaled
dummy_test[:, target_idx] = lstm_test_preds_scaled

# 注意：这里假设你的预处理器里面归一化器叫做 scaler
# 如果报错说 DataPreprocessor 没有 scaler 属性，请根据你实际的变量名修改 (比如 pre.minmax_scaler)
val_preds_real = pre.scaler.inverse_transform(dummy_val)[:, target_idx]
test_preds_real = pre.scaler.inverse_transform(dummy_test)[:, target_idx]

lstm_val_preds = pd.Series(val_preds_real, index=val_idx, name='LSTM')
lstm_test_preds = pd.Series(test_preds_real, index=test_idx, name='LSTM')

# =========================================================================
#  计算指标与保存
# =========================================================================
y_val_true = val.loc[val_idx, 'Close']
y_test_true = test.loc[test_idx, 'Close']

print("\n[Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(y_val_true, lstm_val_preds))

print("\n[Phase 2] 2025 Test Metrics:")
print(calculate_metrics(y_test_true, lstm_test_preds))

# 画预测对比图（2024 Validation）
plotter.plot_predictions_comparison(
    y_true=y_val_true,
    predictions={'LSTM Forecast': lstm_val_preds},
    dates=val_idx,
    title="LSTM 2024 Validation Predictions",
    filename="lstm_val_forecast_2024.png"
)

# 画预测对比图（2025 Test）
plotter.plot_predictions_comparison(
    y_true=y_test_true,
    predictions={'LSTM Forecast': lstm_test_preds},
    dates=test_idx,
    title="LSTM 2025 Test Predictions",
    filename="lstm_test_forecast_2025.png"
)

lstm.save("lstm_model.keras")
lstm_val_preds.to_csv(RESULTS_DIR / 'lstm_val_preds.csv')
lstm_test_preds.to_csv(RESULTS_DIR / 'lstm_test_preds.csv')
print("✅ LSTM execution entirely complete. All files and plots generated!")

E0000 00:00:1774181597.661283  101639 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


--- Training LSTM (Two-Phase Refitting) ---
Training Loss chart saved to: /workspaces/COMP5152ADA_Project_2/reports/figures/lstm_training_loss_60_0.2_5e-4.png

[Phase 1] 2024 Validation Metrics:
{'mse': 1589294.6673014972, 'rmse': 1260.6723076602807, 'mae': 1078.7644439117073, 'mape': 5.344198876370621, 'directional_accuracy': 0.4712041884816754}

[Phase 2] 2025 Test Metrics:
{'mse': 3338389.598402593, 'rmse': 1827.1260488544824, 'mae': 1647.9260413110294, 'mape': 6.934784837746134, 'directional_accuracy': 0.5957446808510638}
✅ LSTM execution entirely complete. All files and plots generated!


## Summary

LSTM model results are saved to `reports/results/lstm_metrics.json`.

Continue to **05_ensemble.ipynb** to combine all model predictions.